# 🧪 LLM Benchmark & Evaluation Suite with LocalTrack

Run automated multi-turn evaluations on candidate models and log evaluation tables, scoring distributions, and artifacts.

In [ ]:
import json
import tempfile
from pathlib import Path
from backend.sdk.localtrack import LocalTrackClient

client = LocalTrackClient(base_url="http://127.0.0.1:8000")
run_id = client.init_run(
    project_name="eval-benchmarks",
    run_name="gsm8k-reasoning-eval",
    config={"eval_dataset": "gsm8k", "temperature": 0.0, "samples": 50},
    tags=["eval", "reasoning", "gsm8k"],
)
print(f"Started evaluation run: {run_id}")

## Run Evaluation Loop & Log Scoring Progression

In [ ]:
benchmark_samples = [
    {"q": "Janet's ducks lay 16 eggs per day...", "correct": True},
    {"q": "A robe takes 2 bolts of blue fiber...", "correct": True},
    {"q": "Josh decides to try flipping a house...", "correct": False},
]

total_correct = 0
for idx, sample in enumerate(benchmark_samples, 1):
    if sample["correct"]:
        total_correct += 1
    accuracy = total_correct / idx
    client.log_metrics({
        "eval/running_accuracy": round(accuracy, 4),
        "eval/samples_processed": idx,
    }, step=idx)

# Upload benchmark summary artifact
with tempfile.NamedTemporaryFile("w", delete=False, suffix=".json") as f:
    json.dump({"dataset": "gsm8k", "total_samples": len(benchmark_samples), "final_accuracy": total_correct / len(benchmark_samples)}, f)
    tmp_name = f.name

client.log_artifact(tmp_name, "eval_summary.json")
Path(tmp_name).unlink(missing_ok=True)
client.finish_run(status="finished")
print("✅ Evaluation complete! Summary logged to LocalTrack.")